# 1. Reduction clause
* [Reduction](https://www.openmp.org/spec-html/5.2/openmpsu107.html#x140-5800002.19.5) is the best way to resolve race conditions where applicable and it often is
* It is applicable when threads altogether calculate $v = v_0 \oplus v_1 \oplus ... \oplus v_{n-1}$ where $v_i$ can be computed independently and $\oplus$ is an associative operator (such as +)
* In serial loop, this could be written by
```
v = initial value;
for (i = 0; i < n; i++) {
  v_i = ...
  v = v + v_i;
}
```
* If we parallelize the above loop, updating $v$ will result in a race condition
* This can be safely parallelized by introducing `reduction(+ : v)`

# <font color="green"> Problem 1 :  Apply reduction</font>
* Add `reduction` clause to `#pragma omp parallel` below and observe the result


In [ ]:
%%writefile omp_reduction.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  int x = 123;
  printf("before : x = %d\n", x);
#pragma omp parallel
  {
    int id = omp_get_thread_num();
    x++;
  }
  printf("after : x = %d\n", x);
  return 0;
}

In [ ]:
%%bash
clang -fopenmp omp_reduction.c -o omp_reduction
# nvc -mp omp_reduction.c -o omp_reduction

In [ ]:
%%bash
OMP_NUM_THREADS=100 ./omp_reduction

# 2. How reduction clause works and why it is preferable when applicable
* Where applicable, reduction is generally much faster than using `#pragma omp atomic` or `#pragma omp critical`
* This is because, internally, each thread computes its partial results using a private variable and combines their results only once at the end
* That is, each update, e.g., `x += expr` updates a thread's private version of variable x instead of updating the shared variable
* Omitting details you can think of what reduction clause is doing is to convert something like

```
int x = G;
#pragma omp parallel reduction(+ : x)
{
   ... x += expr; ...
}
  
```

into something like

```
int x = G;
#pragma omp parallel
{
  int x_priv = 0; // (I) initialize a private version of x
  {
   ... x_priv += expr; ...
  }
#pragma omp atomic
  x += x_priv; // (C) combine the partial results in the private variable into the global variable
}
```
* This is valid because of the associativity of the operation

# 3. User-defined reduction
* Reduction is a general concept for efficiently executing many computations of $v_i$'s in parallel, when the final outcome we wish to compute is $v = v_0 \oplus v_1 \oplus ... \oplus v_{n-1}$ 
* It is applicable whenever the order of combining partial results via $\oplus$ does not affect the final outcome (e.g., +)
* Yet the builtin reduction clause of OpenMP can only specify a few builtin operations for a few builtin types (e.g., int, float, etc.)
* You sometimes desire to apply the efficient execution mechanism of the reduction for more general types (perhaps types you defined)
* [User-defined reduction](https://www.openmp.org/spec-html/5.2/openmpsu107.html#x140-5800002.19.5) exists exactly for that
* You need to define an expression to
  * (C) combine two partial results into one (more specifically, combine a partial result assumed to be in a variable omp_in into another variable omp_out)
  * (I) initialize a thread-private version of the variable to which reduction is applied, named omp_priv
* For example, in the case of the builtin + operator, 
  * (C) would be omp_out += omp_in
  * (I) would be omp_priv = 0

## 3-1. Apply a user-defined reduction
* Here is a simple (broken) parallel for loop that is meant to do a reduction on 3-element vector
* Define a reduction with `#pragma omp declare reduction` and apply it to the parallel loop


In [ ]:
%%writefile omp_ud_reduction.c
#include <stdio.h>
#include <unistd.h>
#include <math.h>
#include <omp.h>

/* 3-element vector */
typedef struct {
  double a[3];
} vec_t;

/* x += y */
void vec_add(vec_t * x, vec_t * y) {
  for (int i = 0; i < 3; i++) {
    x->a[i] += y->a[i];
  }
}

/* x = {0,0,0} */
void vec_init(vec_t * x) {
  for (int i = 0; i < 3; i++) {
    x->a[i] = 0;
  }
}


/* add an appropriate #pragma omp declare reduction ... here */
  
int main() {
  vec_t v;
  vec_init(&v);
  double t0 = omp_get_wtime();
  /* add an appropriate reduction clause, so that
     the result is always {10000,10000,10000} */
#pragma omp parallel for
  for (int i = 0; i < 30000; i++) {
    v.a[i % 3]++;
  }
  double t1 = omp_get_wtime();
  printf("ans = {%.1f, %.1f, %.1f} in %f sec\n", v.a[0], v.a[1], v.a[2], t1 - t0);
  return 0;
}

In [ ]:
%%bash
clang -fopenmp omp_ud_reduction.c -o omp_ud_reduction
# nvc -mp omp_ud_reduction.c -o omp_ud_reduction

In [ ]:
%%bash
OMP_NUM_THREADS=10 ./omp_ud_reduction